# Pragma-Stratified ProbLog: Statistical Validation & Power Analysis

**Experiment Goal:** Validate the hypothesis that classifying linguistic facts by pragmatic tier (assertion, presupposition, implicature) improves neuro-symbolic reasoning accuracy.

**Three validation stages:**
1. **Tier-correctness correlation** (Spearman ρ): Does tier assignment predict fact correctness?
2. **Benchmark comparison** (McNemar test): Does Pragma-Stratified ProbLog outperform Chain-of-Thought?
3. **Hallucination bound analysis** (Pearson r): Can hallucination bound predict reasoning correctness?

**Datasets:** CLUTRR (kinship reasoning) + RuleTaker (rule-based entailment)

**Methods:** RAG baseline, CoT, Flat FOL, Pragma-Stratified ProbLog, LLM-TRes


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('aiohttp==3.11.13')
_pip('rank-bm25==0.2.1')
_pip('loguru==0.8.0')

# Colab core packages (install only locally, use Colab's versions on Colab)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'statsmodels==0.14.6', 'matplotlib==3.10.0')


In [ ]:
import asyncio
import gc
import json
import math
import os
import re
import time
from collections import defaultdict, Counter, deque
from pathlib import Path
from typing import Any

import aiohttp
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

print('All imports successful')


In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-09ca95-pragma-stratified-problog-grounding-hall/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local file")

print("Data loading helper ready")


In [ ]:
data = load_data()
print(f"Loaded data: {data['metadata']['description']}")
print(f"CLUTRR examples: {len(data['datasets'][0]['examples'])}")
print(f"RuleTaker examples: {len(data['datasets'][1]['examples'])}")


In [ ]:
# ─── DEMO CONFIGURATION ─────────────────────────────────────────
# Start with ABSOLUTE MINIMUM values. These control demo scale.
# Increase gradually during testing if results are meaningful.

SAMPLE_CLUTRR = 3      # Use all 3 CLUTRR examples from mini data
SAMPLE_RULETAKER = 3   # Use all 3 RuleTaker examples from mini data
MAX_DEPTH = 4          # BFS max depth for kinship resolution
FW_CHAIN_MAX_ITER = 10 # Forward-chaining max iterations (was 50)

# Tier probability weights (pragmatic stratification)
TIER_PROBS = {"assertion": 0.93, "presupposition": 0.78, "implicature": 0.61, "unknown": 0.75}

print(f"Config: {SAMPLE_CLUTRR} CLUTRR + {SAMPLE_RULETAKER} RuleTaker examples")
print(f"Max depth: {MAX_DEPTH}, Forward-chain iterations: {FW_CHAIN_MAX_ITER}")


In [ ]:
# ─── KINSHIP RELATIONS ──────────────────────────────────────────
RELATIONS = [
    "daughter", "son", "mother", "father", "sister", "brother",
    "wife", "husband", "aunt", "uncle", "niece", "nephew",
    "grandmother", "grandfather", "granddaughter", "grandson",
    "son-in-law", "daughter-in-law",
]
RELATIONS_PAT = "|".join(re.escape(r) for r in sorted(RELATIONS, key=len, reverse=True))

# Tier classification patterns
TIER_PATTERNS = {
    # Assertion: direct explicit kinship
    "assertion": [
        re.compile(r"\[([A-Z][a-z]+)\]'s,?\s+(" + RELATIONS_PAT + r"),?\s+\[([A-Z][a-z]+)\]", re.I),
        re.compile(r"\[([A-Z][a-z]+)\]\s+is\s+(?:the\s+)?(" + RELATIONS_PAT + r")\s+of\s+\[([A-Z][a-z]+)\]", re.I),
    ],
    # Presupposition: possessive pronoun-mediated
    "presupposition": [
        re.compile(r"\[([A-Z][a-z]+)\]\s+and\s+(?:her|his|their)\s+(" + RELATIONS_PAT + r"),?\s+\[([A-Z][a-z]+)\]", re.I),
    ],
    # Implicature: relational verbs / action without explicit naming
    "implicature": [
        re.compile(r"\[([A-Z][a-z]+)\]\s+(?:helped|visited|called|asked|told|met|saw)\s+\[([A-Z][a-z]+)\]", re.I),
    ],
}

KINSHIP_INVERSE = {
    "daughter": "mother", "son": "father", "mother": "daughter", "father": "son",
    "sister": "sister", "brother": "brother", "wife": "husband", "husband": "wife",
    "aunt": "niece", "uncle": "nephew", "niece": "aunt", "nephew": "uncle",
    "grandmother": "granddaughter", "grandfather": "grandson",
    "granddaughter": "grandmother", "grandson": "grandfather",
    "son-in-law": "mother-in-law", "daughter-in-law": "father-in-law",
    "mother-in-law": "son-in-law", "father-in-law": "daughter-in-law",
}

# Kinship composition rules
COMPOSE = {
    ("mother", "daughter"): "sister", ("mother", "son"): "brother",
    ("father", "daughter"): "sister", ("father", "son"): "brother",
    ("mother", "sister"): "aunt", ("mother", "brother"): "uncle",
    ("father", "sister"): "aunt", ("father", "brother"): "uncle",
    ("sister", "mother"): "mother", ("sister", "father"): "father",
    ("brother", "mother"): "mother", ("brother", "father"): "father",
    ("sister", "daughter"): "niece", ("sister", "son"): "nephew",
    ("brother", "daughter"): "niece", ("brother", "son"): "nephew",
    ("daughter", "daughter"): "granddaughter", ("daughter", "son"): "grandson",
    ("son", "daughter"): "granddaughter", ("son", "son"): "grandson",
}

print(f"Loaded {len(RELATIONS)} kinship relations")


In [ ]:
def _add_fact(facts: list, seen: set, a: str, rel: str, b: str, tier: str) -> None:
    """Add a fact and its inverse to the fact list if not already present."""
    key = (a, rel, b)
    if key not in seen:
        seen.add(key)
        facts.append({"a": a, "rel": rel, "b": b, "tier": tier})
    inv = KINSHIP_INVERSE.get(rel)
    if inv:
        ikey = (b, inv, a)
        if ikey not in seen:
            seen.add(ikey)
            facts.append({"a": b, "rel": inv, "b": a, "tier": tier})


def extract_kinship_facts(text: str) -> list[dict]:
    """Extract (entity_a, relation_a_to_b, entity_b, tier) tuples from text."""
    facts: list[dict] = []
    seen: set = set()

    pat1 = re.compile(r"\[([A-Z][a-z]+)\]'s,?\s+(" + RELATIONS_PAT + r"),?\s+\[([A-Z][a-z]+)\]", re.I)
    for m in pat1.finditer(text):
        a, rel, b = m.group(1), m.group(2).lower(), m.group(3)
        _add_fact(facts, seen, a, rel, b, "assertion")

    pat2 = re.compile(r"\[([A-Z][a-z]+)\]\s+is\s+(?:the\s+)?(" + RELATIONS_PAT + r")\s+of\s+\[([A-Z][a-z]+)\]", re.I)
    for m in pat2.finditer(text):
        a, rel, b = m.group(1), m.group(2).lower(), m.group(3)
        _add_fact(facts, seen, b, rel, a, "assertion")

    # Presupposition patterns
    for sent in re.split(r'[.!?]', text):
        entities_in_sent = re.findall(r"\[([A-Z][a-z]+)\]", sent)
        if not entities_in_sent:
            continue
        for m in re.finditer(r"(?:her|his|their)\s+(" + RELATIONS_PAT + r"),?\s+\[([A-Z][a-z]+)\]", sent, re.I):
            rel, b = m.group(1).lower(), m.group(2)
            candidates = [e for e in entities_in_sent if e != b]
            if candidates:
                a = candidates[0]
                _add_fact(facts, seen, a, rel, b, "presupposition")

    # Implicature patterns
    pat_impl = re.compile(
        r"\[([A-Z][a-z]+)\][^.]*?(?:helped|visited|called|asked|told|met|saw|raised)\s+"
        r"(?:her|his|their\s+)?\[([A-Z][a-z]+)\]",
        re.I
    )
    for m in pat_impl.finditer(text):
        a, b = m.group(1), m.group(2)
        key = (a, "related_to", b)
        if key not in seen:
            seen.add(key)
            facts.append({"a": a, "rel": "related_to", "b": b, "tier": "implicature"})

    return facts


def build_kinship_graph(facts: list[dict]) -> dict:
    """Build adjacency: graph[A] = [(rel, B, tier)]"""
    graph = defaultdict(list)
    for f in facts:
        graph[f["a"]].append((f["rel"], f["b"], f["tier"]))
    return graph

print("Kinship extraction functions ready")


In [ ]:
def find_kinship_path(graph: dict, source: str, target: str, use_tier_weights: bool = False) -> tuple[str | None, float, list]:
    """BFS to find kinship relation from source to target. Returns (relation, proof_prob, fact_chain)."""
    if source not in graph and target not in graph:
        return None, 0.0, []

    queue = deque([(source, None, 1.0, [])])
    visited = {source}

    while queue:
        node, current_rel, prob, chain = queue.popleft()
        if len(chain) > MAX_DEPTH:
            continue

        for edge_rel, neighbor, tier in graph[node]:
            if edge_rel == "related_to":
                continue
            edge_prob = TIER_PROBS[tier] if use_tier_weights else 0.5
            new_prob = prob * edge_prob
            new_chain = chain + [(node, edge_rel, neighbor, tier)]

            if current_rel is None:
                composed = edge_rel
            else:
                composed = COMPOSE.get((current_rel, edge_rel))
                if composed is None:
                    composed = f"{current_rel}_{edge_rel}"

            if neighbor == target:
                return composed, new_prob, new_chain

            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, composed, new_prob, new_chain))

    return None, 0.0, []

print("Kinship path finding ready")


In [ ]:
def parse_ruletaker(input_text: str) -> tuple[set, list, str]:
    """Parse context into facts, rules, and question entity/property."""
    ctx_match = re.search(r"Context:\s*(.*?)\s*\n\nQuestion:\s*(.*)", input_text, re.DOTALL)
    if not ctx_match:
        return set(), [], ""
    context = ctx_match.group(1)
    question = ctx_match.group(2).strip().rstrip(".")

    facts: set[tuple[str, str]] = set()
    rules: list[tuple[list[tuple[str, str]], tuple[str, str]]] = []

    for sent in re.split(r'\.\s*', context):
        sent = sent.strip()
        if not sent:
            continue
        
        # Rule pattern: "If someone is A [and B] then they are D"
        rule_m = re.match(
            r"If someone is (\w+(?:\s+\w+)?)(?:\s+and\s+(\w+(?:\s+\w+)?))?(?:\s+and\s+(\w+(?:\s+\w+)?))?\s+then\s+they\s+are\s+(\w+(?:\s+\w+)?)",
            sent, re.I
        )
        if rule_m:
            conditions = []
            for g in [rule_m.group(1), rule_m.group(2), rule_m.group(3)]:
                if g:
                    conditions.append(("?", g.lower().strip()))
            consequent = ("?", rule_m.group(4).lower().strip())
            rules.append((conditions, consequent))
            continue

        # Fact: "X is [not] Y"
        fact_m = re.match(r"(\w+) is (not )?(\w+(?:\s+\w+)?)", sent, re.I)
        if fact_m:
            entity = fact_m.group(1).lower()
            negated = bool(fact_m.group(2))
            prop = fact_m.group(3).lower().strip()
            if not negated:
                facts.add((entity, prop))

    return facts, rules, question


def forward_chain(facts: set, rules: list) -> set:
    """Apply rules until fixed point. Returns extended fact set."""
    facts = set(facts)
    changed = True
    iterations = 0
    while changed and iterations < FW_CHAIN_MAX_ITER:
        changed = False
        iterations += 1
        for conditions, (ce, cp) in rules:
            entities = set(e for e, p in facts)
            for entity in entities:
                satisfied = True
                for (ce2, cp2) in conditions:
                    actual_entity = entity if ce2 == "?" else ce2
                    if (actual_entity, cp2) not in facts:
                        satisfied = False
                        break
                if satisfied:
                    actual_ce = entity if ce == "?" else ce
                    new_fact = (actual_ce, cp)
                    if new_fact not in facts:
                        facts.add(new_fact)
                        changed = True
    return facts

print("RuleTaker parsing ready")


In [ ]:
def ruletaker_pragma_stratified(input_text: str) -> tuple[str, float, float]:
    """Pragma-stratified: tier-weight facts/rules, compute proof probability."""
    facts, rules, question = parse_ruletaker(input_text)
    if not question:
        return "not entailment", 0.5, 1.0

    # Facts are assertions, 1-cond rules are presuppositions, multi-cond are implicatures
    fact_tiers = {f: "assertion" for f in facts}
    rule_tiers = []
    for conditions, consequent in rules:
        tier = "presupposition" if len(conditions) == 1 else "implicature"
        rule_tiers.append(tier)

    # Compute base proof probability from all facts
    p_facts = math.prod(TIER_PROBS[fact_tiers[f]] for f in facts) if facts else 1.0
    p_rules = math.prod(TIER_PROBS[t] for t in rule_tiers) if rule_tiers else 1.0
    base_prob = p_facts * p_rules

    extended_facts = forward_chain(facts, rules)

    neg_m = re.match(r"(\w+) is not (\w+(?:\s+\w+)?)", question, re.I)
    pos_m = re.match(r"(\w+) is (\w+(?:\s+\w+)?)", question, re.I)

    if neg_m:
        entity, prop = neg_m.group(1).lower(), neg_m.group(2).lower().strip()
        entailed = (entity, prop) not in extended_facts
    elif pos_m:
        entity, prop = pos_m.group(1).lower(), pos_m.group(2).lower().strip()
        entailed = (entity, prop) in extended_facts
    else:
        return "not entailment", 0.5, 1.0

    prediction = "entailment" if entailed else "not entailment"

    # Hallucination bound: fraction from non-assertion sources
    n_assertion_facts = len(facts)
    n_presup_rules = sum(1 for t in rule_tiers if t == "presupposition")
    n_implic_rules = sum(1 for t in rule_tiers if t == "implicature")
    total_pieces = n_assertion_facts + n_presup_rules + n_implic_rules
    if total_pieces == 0:
        hb = 0.0
    else:
        hb = (n_presup_rules * 0.5 + n_implic_rules * 1.0) / total_pieces

    return prediction, base_prob, hb

print("Pragma-Stratified reasoning ready")


In [ ]:
def fisher_z_ci(r: float, n: int, alpha: float = 0.05) -> tuple[float, float]:
    """95% CI for correlation via Fisher Z transformation."""
    if n < 4 or abs(r) >= 1.0:
        return (-1.0, 1.0)
    z = 0.5 * math.log((1 + r) / (1 - r))
    se = 1.0 / math.sqrt(n - 3)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    lo_z, hi_z = z - z_crit * se, z + z_crit * se
    lo_r = (math.exp(2 * lo_z) - 1) / (math.exp(2 * lo_z) + 1)
    hi_r = (math.exp(2 * hi_z) - 1) / (math.exp(2 * hi_z) + 1)
    return (round(lo_r, 4), round(hi_r, 4))


def wilson_ci(n_correct: int, n_total: int, alpha: float = 0.05) -> tuple[float, float]:
    if n_total == 0:
        return (0.0, 0.0)
    lo, hi = proportion_confint(n_correct, n_total, alpha=alpha, method="wilson")
    return (round(float(lo), 4), round(float(hi), 4))


def compute_spearman_ci(tier_ordinal: list[int], correctness: list[float]) -> dict:
    """Spearman ρ with 95% CI via Fisher Z."""
    n = len(tier_ordinal)
    if n < 4:
        return {"rho": None, "ci_95": [None, None], "pvalue": None, "n": n}
    if len(set(tier_ordinal)) < 2 or len(set(correctness)) < 2:
        return {"rho": 0.0, "ci_95": [None, None], "pvalue": 1.0, "n": n}
    
    rho, pval = stats.spearmanr(tier_ordinal, correctness)
    rho, pval = float(rho), float(pval)
    ci = fisher_z_ci(rho, n)
    return {
        "rho": round(rho, 4),
        "ci_95": list(ci),
        "pvalue": round(pval, 6),
        "n": n,
    }

print("Statistical functions ready")


In [ ]:
# Extract examples from loaded data
clutrr_exs = data['datasets'][0]['examples'][:SAMPLE_CLUTRR]
rt_exs = data['datasets'][1]['examples'][:SAMPLE_RULETAKER]

print(f"Processing {len(clutrr_exs)} CLUTRR + {len(rt_exs)} RuleTaker examples")
print()

# Derive label map for CLUTRR (from kinship extraction)
label_rel_counts: dict[str, Counter] = defaultdict(Counter)
for ex in clutrr_exs:
    text = ex["input"]
    pair = eval(ex["metadata_entity_pair"])
    label = ex["output"]
    try:
        src, tgt = pair[0], pair[1]
    except Exception:
        continue
    facts = extract_kinship_facts(text)
    graph = build_kinship_graph(facts)
    rel, _, _ = find_kinship_path(graph, src, tgt, use_tier_weights=False)
    if rel and "_" not in rel and rel in RELATIONS:
        label_rel_counts[label][rel] += 1

label_map = {}
for label, counts in label_rel_counts.items():
    if counts:
        label_map[label] = counts.most_common(1)[0][0]

print(f"Derived label map: {dict(list(label_map.items())[:5])}...")


In [ ]:
# Process CLUTRR examples
clutrr_results = []
for ex in clutrr_exs:
    pair = eval(ex["metadata_entity_pair"])
    ea, eb = pair[0], pair[1]
    text = ex["input"]
    gt_label = ex["output"]
    
    facts = extract_kinship_facts(text)
    graph = build_kinship_graph(facts)
    
    # Flat FOL (no tier weighting)
    rel_flat, prob_flat, chain_flat = find_kinship_path(graph, ea, eb, use_tier_weights=False)
    
    # Pragma-Stratified (with tier weighting)
    rel_prag, prob_prag, chain_prag = find_kinship_path(graph, ea, eb, use_tier_weights=True)
    
    clutrr_results.append({
        "input": text[:100] + "...",
        "ground_truth": gt_label,
        "entities": f"{ea} -> {eb}",
        "flat_fol_rel": rel_flat,
        "flat_fol_prob": round(prob_flat, 3),
        "pragma_rel": rel_prag,
        "pragma_prob": round(prob_prag, 3),
    })

print(f"\n=== CLUTRR Results (n={len(clutrr_results)}) ===")
for i, r in enumerate(clutrr_results):
    print(f"\nExample {i+1}:")
    print(f"  Entities: {r['entities']}")
    print(f"  Ground truth label: {r['ground_truth']}")
    print(f"  Flat FOL: {r['flat_fol_rel']} (prob={r['flat_fol_prob']})")
    print(f"  Pragma-Stratified: {r['pragma_rel']} (prob={r['pragma_prob']})")


In [ ]:
# Process RuleTaker examples
rt_results = []
tier_analysis = []

for ex in rt_exs:
    text = ex["input"]
    gt_label = ex["output"]
    config = ex.get("metadata_config", "unknown")
    
    # Pragma-Stratified reasoning
    pred, prob, hb = ruletaker_pragma_stratified(text)
    
    rt_results.append({
        "config": config,
        "input": text[:100] + "...",
        "ground_truth": gt_label,
        "prediction": pred,
        "prob": round(prob, 3),
        "hallucination_bound": round(hb, 3),
        "correct": pred == gt_label,
    })
    
    tier_analysis.append({
        "config": config,
        "hb": hb,
        "correct": 1.0 if pred == gt_label else 0.0,
    })

print(f"\n=== RuleTaker Results (n={len(rt_results)}) ===")
for i, r in enumerate(rt_results):
    status = "✓" if r['correct'] else "✗"
    print(f"\n{status} Example {i+1} ({r['config']}):")
    print(f"  Ground truth: {r['ground_truth']}")
    print(f"  Prediction: {r['prediction']}")
    print(f"  Proof probability: {r['prob']}")
    print(f"  Hallucination bound: {r['hallucination_bound']}")


In [ ]:
# Summary statistics
print("\n" + "="*60)
print("DEMO RESULTS SUMMARY")
print("="*60)

clutrr_correct = sum(1 for r in clutrr_results if r['pragma_rel'] is not None)
rt_correct = sum(1 for r in rt_results if r['correct'])

print(f"\nKinship (CLUTRR):")
print(f"  Examples with valid paths: {clutrr_correct}/{len(clutrr_results)}")
print(f"  Avg pragma prob: {np.mean([r['pragma_prob'] for r in clutrr_results]):.3f}")

print(f"\nEntailment (RuleTaker):")
print(f"  Correct predictions: {rt_correct}/{len(rt_results)}")
print(f"  Avg proof probability: {np.mean([r['prob'] for r in rt_results]):.3f}")
print(f"  Avg hallucination bound: {np.mean([r['hallucination_bound'] for r in rt_results]):.3f}")

# Tier-correctness analysis
if len(tier_analysis) >= 4:
    tiers_ord = [2 for _ in tier_analysis]  # Simplified: treat all as presupposition
    corrs = [t['correct'] for t in tier_analysis]
    tier_corr = compute_spearman_ci(tiers_ord, corrs)
    print(f"\nTier-Correctness (Spearman ρ):")
    print(f"  ρ = {tier_corr['rho']}")
    print(f"  p-value = {tier_corr['pvalue']}")
    print(f"  n = {tier_corr['n']}")

print(f"\nConfiguration:")
print(f"  CLUTRR examples: {SAMPLE_CLUTRR}")
print(f"  RuleTaker examples: {SAMPLE_RULETAKER}")
print(f"  Max BFS depth: {MAX_DEPTH}")
print(f"  Forward-chain iterations: {FW_CHAIN_MAX_ITER}")
print(f"\n(Increase config parameters to scale up the demo)")


In [ ]:
# Create visualization of hallucination bound vs correctness
if len(tier_analysis) > 1:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    
    hbs = [t['hb'] for t in tier_analysis]
    corrs = [t['correct'] for t in tier_analysis]
    configs = [t['config'] for t in tier_analysis]
    
    colors = ['green' if c == 1.0 else 'red' for c in corrs]
    ax.scatter(hbs, corrs, c=colors, s=100, alpha=0.6)
    
    ax.set_xlabel('Hallucination Bound (HB)')
    ax.set_ylabel('Correctness (0=incorrect, 1=correct)')
    ax.set_title('Pragma-Stratified ProbLog: HB vs Correctness')
    ax.set_ylim(-0.1, 1.1)
    ax.grid(True, alpha=0.3)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='green', alpha=0.6, label='Correct'),
                       Patch(facecolor='red', alpha=0.6, label='Incorrect')]
    ax.legend(handles=legend_elements, loc='best')
    
    plt.tight_layout()
    plt.savefig('hb_correctness_demo.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Plot saved as 'hb_correctness_demo.png'")
